# ✈️ Airline Passenger Satisfaction Prediction (Top 15% / Score: 0.960+)

## 🎯 Project Overview
This notebook presents my solution for the Kaggle Playground Series (Season 6, Episode 10) - Predicting Airline Satisfaction. The evaluation metric is **ROC-AUC**.

## 🧠 Strategy Summary
1. **Exploratory Data Analysis (EDA)**: Identified the zero-inflated nature of delays and the extreme dominance of `Class` and `Type of Travel`.
2. **Advanced Feature Engineering**: Focused on feature interactions and specifically engineered **Route Geometry** features to capture flight-level context.
3. **Modeling**: Trained LightGBM, XGBoost (High-Bin), and CatBoost using 5-Fold Stratified Cross-Validation.
4. **Ensembling**: Combined models using Rank Average and Weighted Blend to maximize the leaderboard score.

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from scipy.stats import rankdata

import seaborn as sns
import matplotlib.pyplot as plt
import time

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

from sklearn.ensemble import StackingClassifier

import xgboost as xgb
import lightgbm as lgb

from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import log_loss,classification_report, roc_auc_score, ConfusionMatrixDisplay, confusion_matrix


from sklearn.linear_model import Ridge

import warnings
warnings.filterwarnings("ignore")

## 1. Data Loading & Preprocessing

In [ ]:
#Reading CSV file of dataset
train=pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/train.csv')
test=pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/test.csv')
sample=pd.read_csv('/kaggle/input/competitions/playground-series-s6e10/sample_submission.csv')

In [ ]:
train.head()

In [ ]:
train.columns

In [ ]:
train.drop('id', axis=1, inplace=True)

In [ ]:
train.info()
train.isnull().sum()

### Data Loading & Structural Overview

We load the three provided files: `train.csv`, `test.csv`, and `sample_submission.csv`.

A first structural inspection of the training set (`train.info()`) reveals the following:

- **Shape**: 699,635 rows × 23 columns — a large tabular dataset (~145 MB in memory).
- **Target**: `satisfaction` is already stored as `bool`, so it can be cast to `int` directly for AUC evaluation.
- **Numeric features**: 14 ordinal service-rating columns (`Inflight wifi service`, `Online boarding`, ..., `Cleanliness`) stored as `int64`, plus `Age`, `Flight Distance`, and `Departure Delay in Minutes`.
- **Categorical features**: 4 `str` columns — `Gender`, `Customer Type`, `Type of Travel`, `Class`.
- **Identifier**: `id` (no predictive power — will be dropped before modeling).
- **Missing values**: only `Arrival Delay in Minutes` contains NaNs (292 rows, i.e. ~0.042% of the data). Everything else is fully populated.

The large row count and the clean structure suggest that a gradient-boosted tree baseline (LightGBM / CatBoost) will converge quickly, and that feature engineering will play a larger role than hyper-parameter tuning.

### Handling Missing Values

`train.isnull().sum()` shows a single column with missing entries:

| Column | Missing | % of rows |
|---|---|---|
| `Arrival Delay in Minutes` | 292 | ≈ 0.042 % |

Given that (a) the missing fraction is negligible and (b) `Departure Delay in Minutes` and `Arrival Delay in Minutes` are extremely highly correlated in this dataset (departure delay is almost always equal to or slightly smaller than arrival delay), we impute `Arrival Delay in Minutes` **using the corresponding `Departure Delay in Minutes` value** rather than the global median.

Rationale:
- The global median of both delay columns is **0** (most flights depart and arrive on time), which would silently mislabel the delayed passengers in those 292 rows.
- Filling from the departure delay keeps the *physical consistency* between the two delay features, which is important because later we will derive `total_delay`, `delay_diff`, and `is_delayed` flags from them.
- Test-set imputation follows the **exact same rule** using its own `Departure Delay in Minutes` values (no leakage from train).

The same imputation is applied to `test` to keep the pipeline consistent.

In [ ]:
# Impute Arrival Delay using Departure Delay (they are almost identical in this dataset)
mask = train['Arrival Delay in Minutes'].isna()
# Quick sanity: correlation on non-null rows
corr = train[~mask][['Departure Delay in Minutes','Arrival Delay in Minutes']].corr().iloc[0,1]
print(corr)

# Since 84% of rows have zero departure delay and arrival delay is usually equal,
# simply fill with Departure Delay value. This is safer than global median (which would be 0 anyway)
train['Arrival Delay in Minutes'] = train['Arrival Delay in Minutes'].fillna(
    train['Departure Delay in Minutes']
)
test['Arrival Delay in Minutes'] = test['Arrival Delay in Minutes'].fillna(
    test['Departure Delay in Minutes']
)

In [ ]:
train.describe().T

### Descriptive Statistics

`train.describe().T` reveals several structural facts that will shape our feature engineering.

#### Identifier
- `id` is a simple sequence from 0 to 699,634 (mean ≈ std × √3 / √12, i.e. uniform). It carries **no predictive information** and will be dropped before modeling.

#### Numeric continuous features
- **`Age`**: range 7–85, mean 39.0, std 13.8, quartiles [27, 40, 50]. Symmetric and well-behaved, no obvious outliers, no transformation needed.
- **`Flight Distance`**: range 67–4,983, mean 1,354 > median 954. **Right-skewed** with a long upper tail; candidate for a `log1p` transform or binning.

#### Ordinal service ratings (14 columns)
- All columns are on a 1–5 scale, but **almost all of them have a minimum of 0**, with the single exception of `Baggage handling` (min = 1). This suggests that `0` in these columns may encode "service not used / not rated" rather than a genuine rating — this will be verified in the next cell with a per-column count of zeros.
- Mean ratings per service (descending): `Baggage handling` (3.81) > `Seat comfort` (3.57) > `On-board service` (3.52) > `Leg room service` (3.47) > `Inflight entertainment` (3.45) > `Checkin service` (3.42) > `Cleanliness` (3.38) > `Online boarding` (3.36) > `Food and drink` (3.27) > `Departure/Arrival time convenient` (3.16) > `Gate location` (3.01) > `Ease of Online booking` (2.81) > `Inflight wifi service` (2.78). So on average, wifi and online booking are the **lowest-rated** services, while baggage handling is the highest.
- Standard deviations are very homogeneous across services (≈ 1.1–1.5), which hints that they may share similar variance and possibly be collinear.

#### Delay features
- **`Departure Delay in Minutes`**: mean 1.18, std 5.98, **75 % quantile = 0**, max 489.
- **`Arrival Delay in Minutes`**: mean 1.14, std 5.77, **75 % quantile = 0**, max 491.
- Both are **heavily zero-inflated** (≥ 75 % of flights have zero delay) with a long heavy tail extending up to ~8 hours. This strongly motivates:
  - a binary `is_delayed` flag,
  - a `log1p`-transformed delay value,
  - a derived `delay_recovery = Departure Delay − Arrival Delay` and `total_delay = Departure + Arrival`.

#### Takeaway
No impossible values were found for `Age` or `Flight Distance`. The most important structural finding is that the delay features need monotone + log-style transformations, and that the presence of `0` in the service-rating columns has to be investigated before they are treated as clean 1–5 ordinal scales.

In [ ]:
service_cols = ['Inflight wifi service','Departure/Arrival time convenient',
 'Ease of Online booking','Gate location','Food and drink','Online boarding',
 'Seat comfort','Inflight entertainment','On-board service','Leg room service',
 'Baggage handling','Checkin service','Cleanliness']

# Count and percentage of zeros per service column
zero_counts = (train[service_cols] == 0).sum().sort_values(ascending=False)
zero_pct = (train[service_cols] == 0).mean().sort_values(ascending=False) * 100

pd.DataFrame({'zero_count': zero_counts, 'zero_pct': zero_pct.round(3)})

In [ ]:
train.nunique()

### Cardinality Check

`train.nunique()` confirms the structure of each feature group:

- **Categorical features (low cardinality)**: `Gender` (2), `Customer Type` (2), `Type of Travel` (2), and `Class` (3). These are trivial to encode — for tree models, a simple `category` dtype or label encoding is sufficient; one-hot is also viable but not necessary.
- **Service-rating features**: all 14 columns take 6 unique values (0–5), except `Baggage handling` which takes 5 (1–5) — consistent with the earlier finding that baggage handling has no zero values.
- **Continuous features**: `Age` (75 unique), `Flight Distance` (3,474 unique), and the two delay columns (167 and 168 unique values, still heavily zero-inflated as shown in the describe step).

## 2. Data Visualization

In [ ]:
print(train['satisfaction'].value_counts(normalize=True))

sns.countplot(x='satisfaction', data=train , palette='Set1')
plt.title('Target Distribution'); plt.show()

### Target Distribution

`satisfaction` is split into **55.6 % False** vs **44.4 % True** — mildly imbalanced but not enough to require any resampling, class weighting, or threshold tuning. The metric is AUC, which is insensitive to class priors, so we proceed with the raw class proportions and simply cast the boolean target to `int` for training.

In [ ]:
for col in ['Gender', 'Customer Type', 'Type of Travel', 'Class']:
    pd.crosstab(train[col], train['satisfaction'], normalize='index').plot.bar(stacked=True)
    plt.title(col); plt.show()

### Categorical Features vs Target

Stacked bar charts of satisfaction rate (normalized by feature value) reveal clear differences in predictive power:

- **`Gender`** — Female and Male show nearly identical satisfaction rates (~44 %). Weak feature on its own.
- **`Customer Type`** — Loyal customers are satisfied ~50 % of the time, versus only ~20 % for disloyal customers. Strong signal.
- **`Type of Travel`** — Business travelers are satisfied ~60 % of the time, versus only ~10 % for personal travelers. Very strong signal.
- **`Class`** — The strongest of the four: Business class satisfaction is ~72 %, while Eco is only ~18 % and Eco Plus ~25 %.

**Takeaway**: `Class`, `Type of Travel`, and `Customer Type` are the dominant categorical drivers, and they are almost certainly correlated (Business class ↔ Business travel ↔ Loyal customer). The interaction between `Class` and `Type of Travel` is likely to be especially informative and should be considered during feature engineering.

In [ ]:
service_cols = ['Inflight wifi service','Departure/Arrival time convenient',
 'Ease of Online booking','Gate location','Food and drink','Online boarding',
 'Seat comfort','Inflight entertainment','On-board service','Leg room service',
 'Baggage handling','Checkin service','Cleanliness']

# Distribution of each service rating by target
fig, axes = plt.subplots(4, 4, figsize=(18,14))
for ax, col in zip(axes.flat, service_cols):
    sns.countplot(x=col, hue='satisfaction', data=train, ax=ax, palette='Set1')
    ax.set_title(col, fontsize=9)
plt.tight_layout(); plt.show()

### Service Ratings vs Target

We visualize the distribution of satisfaction for each of the 14 ordinal service ratings.

**Highly discriminative features:**
- `Online boarding`, `Inflight entertainment`, and `Seat comfort` show the strongest class separation. For these, ratings of 1–3 are heavily dominated by unsatisfied passengers (False), while ratings of 4–5 flip sharply towards satisfied passengers (True). These are likely to be the primary drivers of satisfaction.

**Weakly discriminative features:**
- `Gate location` and `Departure/Arrival time convenient` show almost identical distributions for both classes. They add very little predictive power on their own and might even be dropped without much loss, though tree models can safely ignore them.

**Mid-tier features:**
- `Cleanliness`, `On-board service`, `Leg room service`, `Baggage handling`, and `Inflight wifi service` show a clear but less dramatic separation.

**Implication for Feature Engineering:**
Because several service columns share a similar monotonic relationship with the target, aggregating them is likely to be beneficial. We will construct a `mean_top_services` feature using the top discriminators (`Online boarding`, `Inflight entertainment`, `Seat comfort`), as a combined signal may outperform any single column.

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(14,5))
sns.kdeplot(data=train, x='Age', hue='satisfaction', fill=True, ax=axes[0], palette='Set1')
sns.kdeplot(data=train, x='Flight Distance', hue='satisfaction', fill=True, ax=axes[1], palette='Set1')
plt.show()

In [ ]:
# Create a figure with 1 row and 2 columns
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot for Age (handling Seaborn's palette warning by using hue and legend=False)
sns.boxplot(x='satisfaction', y='Age', data=train, palette='Set1', hue='satisfaction', legend=False, ax=axes[0])
axes[0].set_title('Age vs Satisfaction')

# Boxplot for Flight Distance
sns.boxplot(x='satisfaction', y='Flight Distance', data=train, palette='Set1', hue='satisfaction', legend=False, ax=axes[1])
axes[1].set_title('Flight Distance vs Satisfaction')

plt.tight_layout()
plt.show()

### Continuous Features vs Target: Age and Flight Distance

**Age**
The KDE plot shows a clear shift in distribution: unsatisfied passengers peak in their mid-20s, while satisfied passengers peak around 40–50 years old. The boxplot confirms this — the median age for satisfied passengers is ~43, compared to ~35 for unsatisfied ones. Age is a moderately strong, well-behaved feature.

**Flight Distance**
This feature provides a strong separation between classes. Unsatisfied passengers are overwhelmingly concentrated on short flights (median ~800 km), while satisfied passengers take significantly longer flights (median ~1600 km). The KDE plot for the `True` class is distinctly shifted to the right, with multiple peaks corresponding to longer-haul routes. The large upper tail of the `False` class suggests some long-haul passengers are still dissatisfied, but the bulk of short-haul passengers are not.

**Takeaway**
Both features add meaningful predictive signal. Because `Flight Distance` correlates strongly with travel purpose and cabin class, tree-based models will naturally capture any interactions. `log1p` transformation of `Flight Distance` may slightly help linear models but is not strictly necessary for gradient boosting.

In [ ]:
print((train['Departure Delay in Minutes']==0).mean())
print((train['Arrival Delay in Minutes']==0).mean())

# log distribution
train['dep_log'] = np.log1p(train['Departure Delay in Minutes'])
sns.kdeplot(data=train, x='dep_log', hue='satisfaction', fill=True)

### Delay Features: Zero-Inflation and Distribution

The KDE plot of log-transformed departure delay (`dep_log`) confirms a severe **zero-inflation**: the vast majority of flights (~75%) have zero delay, producing a massive spike at 0. 

Crucially, the curves for satisfied (`True`) and unsatisfied (`False`) passengers **overlap almost perfectly** at zero and decay at similar rates. This indicates that raw delay magnitude — even after `log1p` transformation — has weak standalone discriminative power for predicting satisfaction.

**Implications for Feature Engineering:**
- Instead of relying purely on the raw or log-transformed delay, we will create a binary **`is_delayed`** flag (delay > 0), and possibly an extreme-delay threshold (e.g., > 30 mins), which may carry non-linear signal.
- Since delay alone doesn't separate classes well, the model will likely learn to rely more heavily on service ratings, cabin class, and interactions (e.g., a delayed *business* passenger vs. a delayed *eco* passenger).

In [ ]:
# correlation between numeric + ordinal only
corr = train.select_dtypes(include='number').corr()
plt.figure(figsize=(14,10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=True)
plt.show()

# correlation with target only
train['target'] = train['satisfaction'].astype(int)
train.select_dtypes('number').corr()['target'].sort_values(ascending=False)

### Correlation Analysis

**Correlation with Target:**
- The strongest positive correlates of satisfaction are `Online boarding` (0.56), `Inflight entertainment` (0.43), `Seat comfort` (0.40), and `Flight Distance` (0.37).
- `Gate location` (0.02) and `Departure/Arrival time convenient` (-0.05) have virtually no linear relationship with the target.
- Both delay features and their log-transformed version show a surprisingly weak negative correlation with the target (around -0.04). This confirms our previous KDE finding that delay magnitude alone is a poor linear predictor of satisfaction.

**Multicollinearity:**
- There are distinct clusters of highly correlated features:
  - **Seat/Entertainment/Cleanliness cluster**: `Seat comfort`, `Inflight entertainment`, and `Cleanliness` are strongly correlated (up to 0.76).
  - **Booking/Wifi cluster**: `Inflight wifi service`, `Ease of Online booking`, and `Departure/Arrival time convenient` are highly inter-correlated (up to 0.77).
  - **Delay cluster**: `Departure Delay` and `Arrival Delay` are heavily correlated (0.84).
- Also, `Food and drink` and `Cleanliness` show a notable correlation (0.68).

**Implications for Modeling:**
While gradient-boosted trees (LightGBM, CatBoost) are robust to multicollinearity, this confirms that aggregating highly correlated features (e.g., a combined "comfort rating") or dropping redundant columns may slightly simplify the model and reduce overfitting risk.

In [ ]:
# Class × Type of Travel
sns.catplot(x='Class', hue='satisfaction', col='Type of Travel',
            data=train, kind='count', height=4, aspect=1, palette='Set1')

## Interaction Effect: Class × Type of Travel

This catplot visualizes the joint distribution of `Class` and `Type of Travel` broken down by the target. 

**Key Observations:**
- **The "Golden Combination"**: Passengers flying for **Business travel** in **Business class** are overwhelmingly satisfied. This single group accounts for the vast majority of all `True` (satisfied) labels in the dataset.
- **The "Unsatisfied Majority"**: Passengers traveling for **Personal travel** in **Eco** class are overwhelmingly unsatisfied. 
- **Asymmetry**: While Business travel in Eco class is still largely dissatisfied, the sheer volume of Business class passengers in Business travel completely flips the overall satisfaction ratio for that segment.

**Takeaway for Feature Engineering:**
This confirms a massive interaction effect. Neither `Class` nor `Type of Travel` tells the full story on its own. We will create an explicit interaction feature (e.g., `Class_Type` by concatenating the two) to help the model capture this specific combination effortlessly.

## 3. Feature Engineering & Route Geometry

Standard GBDT models struggle with aggregating local context. Because flight distance acts as an exact proxy for route equipment (aircraft type, crew, etc.), we engineered **Route Geometry** features.

- **Service Aggregations**: `weighted_top3`, `mean_top_services`, `num_zero_ratings`, `num_high_ratings`.
- **Route Geometry**: Grouped the combined train and test data by `Flight Distance` and calculated the mean ratings across all 13 services, mean age, flight count, and the share of Business class passengers per route.
- **Delay Dynamics**: Dropped raw delay magnitudes as EDA showed they add almost zero signal. Replaced with binary flags like `is_delayed`.

In [ ]:
def feature_engineering(df):
    df = df.copy()
    
    # 1. Cast categories first
    cat_cols = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
    for col in cat_cols:
        df[col] = df[col].astype('category')
        
    # 2. Service Columns List
    service_cols = ['Inflight wifi service','Departure/Arrival time convenient',
     'Ease of Online booking','Gate location','Food and drink','Online boarding',
     'Seat comfort','Inflight entertainment','On-board service','Leg room service',
     'Baggage handling','Checkin service','Cleanliness']
    
    # 3. Keep the top-performing engineered features
    df['weighted_top3'] = (df['Online boarding'] * 0.56 +
                           df['Inflight entertainment'] * 0.43 +
                           df['Seat comfort'] * 0.40) / (0.56 + 0.43 + 0.40)
    
    df['mean_top_services'] = df[['Online boarding', 'Inflight entertainment', 'Seat comfort']].mean(axis=1)
    
    df['Class_Type'] = df['Class'].astype(str) + '_' + df['Type of Travel'].astype(str)
    
    # 4. Rating counts
    df['num_zero_ratings'] = (df[service_cols] == 0).sum(axis=1)
    df['num_high_ratings'] = (df[service_cols] >= 4).sum(axis=1)
    
    # 5. Max rating
    df['service_max'] = df[service_cols].max(axis=1)
    
    # 6. Continuous Transformation & Interaction
    df['log_flight_distance'] = np.log1p(df['Flight Distance'])
    df['age_x_class'] = (df['Age'] * (df['Class'].cat.codes + 1)).astype('int16')
    
    # 7. Finalize categorical columns
    cat_cols_final = ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'Class_Type']
    for col in cat_cols_final:
        df[col] = df[col].astype('category')
        
    df = df.drop(columns=['id'], errors='ignore')
    
    return df

In [ ]:
# Apply and rebuild X, y, X_test
train_fe = feature_engineering(train)
test_fe = feature_engineering(test)

# 1. Define target variable
y = train_fe['satisfaction'].astype(int)

# 2. Drop target columns from train and create X
# We drop 'satisfaction' and 'target' (the temporary column from correlation)
X = train_fe.drop(columns=['satisfaction', 'target','dep_log'], errors='ignore')
X_test = test_fe.copy()

# 3. Verify alignment
print(f"X shape:      {X.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Are feature columns identical? {X.columns.equals(X_test.columns)}")

In [ ]:
print("Computing Route Geometry...")
# Use raw train and test to avoid any leakage
combined = pd.concat([
    train.drop(columns=['satisfaction', 'id'], errors='ignore'),
    test.drop(columns=['id'], errors='ignore')
], ignore_index=True)

service_cols = ['Inflight wifi service','Departure/Arrival time convenient',
 'Ease of Online booking','Gate location','Food and drink','Online boarding',
 'Seat comfort','Inflight entertainment','On-board service','Leg room service',
 'Baggage handling','Checkin service','Cleanliness']

g = combined.groupby('Flight Distance')
agg = g[service_cols + ['Age']].mean().add_prefix('fd_mean_')
agg['fd_count'] = g.size().astype(np.int32)

if 'Class' in combined.columns:
    is_business = (combined['Class'] == 'Business').astype(int)
    agg['fd_share_Business'] = is_business.groupby(combined['Flight Distance']).mean()

# --- 3. Join Route Geometry to X and X_test ---
X = X.join(agg, on='Flight Distance', rsuffix='_route')
X_test = X_test.join(agg, on='Flight Distance', rsuffix='_route')

# --- 4. Final Verification ---
print(f"✅ Route Geometry added!")
print(f"X shape:      {X.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Columns match: {X.columns.equals(X_test.columns)}")

## 4. Building and Training the Model

We trained three state-of-the-art gradient boosting models with 5-Fold Cross-Validation:
1. **LightGBM**: Native categorical handling, robust baseline.
2. **XGBoost (High-Bin)**: Tuned with `max_bin=4096` and `colsample_bytree=0.2` to capture fine-grained continuous splits.
3. **CatBoost**: Strong performance with categorical interactions.

*Note: Training was accelerated using GPU (T4) with `border_count=254` for CatBoost to match CPU performance.*

### LightGBM

In [ ]:
# 1. Setup Cross-Validation
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Arrays to store predictions
lgb_oof_preds = np.zeros(len(X))
lgb_test_preds = np.zeros(len(X_test))

# LightGBM parameters (Baseline)
# Best parameters from RandomizedSearchCV
lgb_params =  {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.043013213230530575,
    'num_leaves': 39,
    'max_depth': 10,
    'min_data_in_leaf': 148,
    'feature_fraction': 0.7113068222276344,
    'bagging_fraction': 0.8356740207895621,
    'bagging_freq': 1,
    'lambda_l1': 0.011756010900231853,
    'lambda_l2': 0.05731044951044767,
    'max_bin': 255,
    'num_threads': -1,
    'seed': 42,
    'verbosity': -1
}
# Also increase num_boost_round to 3000-5000 since lr is lower


# 2. Training Loop
print(f"Starting {n_splits}-Fold Cross-Validation...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    # Split data
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    
    # Create LightGBM datasets
    dtrain = lgb.Dataset(X_train, label=y_train)
    dval = lgb.Dataset(X_val, label=y_val, reference=dtrain)
    
    # Train model with early stopping
    lgb_model = lgb.train(
        lgb_params,
        dtrain,
        num_boost_round=4000,
        valid_sets=[dtrain, dval],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(period=200)]
    )
    
    # Predict OOF and Test
    lgb_oof_preds[val_idx] = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
    lgb_test_preds += lgb_model.predict(X_test, num_iteration=lgb_model.best_iteration) / n_splits
    
    # Print fold score
    fold_auc = roc_auc_score(y_val, lgb_oof_preds[val_idx])
    print(f"Fold {fold+1} AUC: {fold_auc:.5f}")

# 3. Overall CV Score
overall_auc = roc_auc_score(y, lgb_oof_preds)
print(f"\n=========================================")
print(f"Overall OOF AUC: {overall_auc:.5f}")
print(f"=========================================")

# 4. Create Submission File
# Note: We use the original test dataframe's 'id' because we dropped it in the FE step
submission = pd.DataFrame({
    'id': test['id'],
    'satisfaction': lgb_test_preds
})

submission.to_csv('submission_baseline.csv', index=False)
print("\nSubmission saved to 'submission_baseline.csv'")
print(submission.head())

In [ ]:
# Train one model on full data for importance analysis
full_model = lgb.train(
    lgb_params,
    lgb.Dataset(X, label=y),
    num_boost_round=1000,
)

# Plot importance
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 12))
lgb.plot_importance(full_model, ax=ax, max_num_features=40, importance_type='gain')
plt.tight_layout()
plt.show()

In [ ]:
# Print sorted importance
importance_df = pd.DataFrame({
    'feature': full_model.feature_name(),
    'gain': full_model.feature_importance(importance_type='gain'),
}).sort_values('gain', ascending=False)
print(importance_df.to_string())

### Feature Importance Analysis (LightGBM)

The feature importance plot (Gain) reveals three key insights about our pipeline:

#### 1. Engineered Features Outperform Raw Data
Our custom aggregations are highly effective. **`weighted_top3`** and **`mean_top_services`** rank 2nd and 5th respectively, proving that compressing the top discriminative services into single features helps the tree models significantly. **`Class_Type`** (rank 6) also confirms the importance of capturing categorical interactions.

#### 2. The Power of Route Geometry (Collective Impact)
While individual `fd_mean_*` features appear in the lower half of the plot (e.g., `fd_mean_Inflight_wifi_service`), their sheer number provides crucial **contextual anchors**. The model uses these route-level aggregates to refine its leaf values, which is the primary reason our CV jumped from ~0.9588 to ~0.9602. 

#### 3. Irrelevant Features (Noise)
- **Delay Features**: `Arrival_Delay` and `Departure_Delay` sit at the very bottom (Gain < 8,000), confirming our EDA that raw delay magnitudes carry almost no signal in this synthetic dataset.
- **Demographics**: `Gender` and `log_flight_distance` are virtually ignored by the model.

**Takeaway**: Future iterations could drop the delay and gender columns entirely to slightly speed up training and reduce noise without any loss in AUC.

### XGBoost

In [ ]:
# Prepare data for XGBoost (supports native category)
# We use the clean X and X_test from the latest feature engineering step
xgb_oof_preds = np.zeros(len(X))
xgb_test_preds = np.zeros(len(X_test))

xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'enable_categorical': True,  # Enable native categorical handling
    'learning_rate': 0.05,
    'max_depth': 7,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_jobs': -1,
    'random_state': 42,
}

xgb_params_v2 = {
    'n_estimators': 1200,
    'learning_rate': 0.04,
    'max_depth': 8,
    'max_bin': 4096,
    'subsample': 0.80,
    'colsample_bytree': 0.20,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'random_state': 42,
    'enable_categorical': True, # Enable native categorical handling for pandas 'category' dtype columns
    'n_jobs': -1,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Starting XGBoost 5-Fold CV...\n")
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    dtrain = xgb.DMatrix(X.iloc[train_idx], label=y.iloc[train_idx], enable_categorical=True)
    dval   = xgb.DMatrix(X.iloc[val_idx], label=y.iloc[val_idx], enable_categorical=True)
    dtest  = xgb.DMatrix(X_test, enable_categorical=True)
    
    xgb_model = xgb.train(
        xgb_params_v2,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=200
    )
    
    #Predict OOF and Test
    xgb_oof_preds[val_idx] = xgb_model.predict(dval, iteration_range=(0, xgb_model.best_iteration + 1))
    xgb_test_preds   += xgb_model.predict(dtest, iteration_range=(0, xgb_model.best_iteration + 1)) / 5
    print(f"Fold {fold+1} AUC: {roc_auc_score(y.iloc[val_idx], xgb_oof_preds[val_idx]):.5f}\n")


print(f"=========================================")
print(f"XGBoost Overall OOF AUC: {roc_auc_score(y, xgb_oof_preds):.5f}")
print(f"=========================================")

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'satisfaction': xgb_test_preds
})

submission.to_csv('submission_xgb.csv', index=False)
print("\nSubmission saved to 'submission_xgb.csv'")
print(submission.head())

### CatBoost

In [ ]:
# ---- 3. CatBoost ----
# CatBoost needs string for categorical features
cat_features = ['Gender', 'Customer Type', 'Type of Travel', 'Class', 'Class_Type']
X_cb = X.copy()
X_test_cb = X_test.copy()

for col in cat_features:
    X_cb[col] = X_cb[col].astype(str)
    X_test_cb[col] = X_test_cb[col].astype(str)

cb_oof_preds = np.zeros(len(X_cb))
cb_test_preds = np.zeros(len(X_test_cb))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Starting CatBoost 5-Fold CV...\n")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_cb, y)):
    cb_model = CatBoostClassifier(
        iterations=3000,
        task_type='GPU',
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        eval_metric='AUC',
        cat_features=cat_features,
        early_stopping_rounds=150,
        verbose=200,
        random_seed=42,
        thread_count=-1,
    )
    
    cb_model.fit(X_cb.iloc[train_idx], y.iloc[train_idx], eval_set=(X_cb.iloc[val_idx], y.iloc[val_idx]), use_best_model=True)
    
    cb_oof_preds[val_idx] = cb_model.predict_proba(X_cb.iloc[val_idx])[:, 1]
    cb_test_preds += cb_model.predict_proba(X_test_cb)[:, 1] / 5
    
    print(f"Fold {fold+1} AUC: {roc_auc_score(y.iloc[val_idx], cb_oof_preds[val_idx]):.5f}\n")

print(f"=========================================")
print(f"CatBoost Overall OOF AUC: {roc_auc_score(y, cb_oof_preds):.5f}")
print(f"=========================================")

## 5. Ensembling (Rank Average & Weighted Blend)

To reduce variance and leverage the diversity of the models, we blended their Out-Of-Fold (OOF) predictions.
- **Rank Average**: Converted probabilities to ranks to normalize scale differences.
- **Weighted Blend**: Optimized weights using a grid search on the OOF predictions.

The final submission uses the Weighted Blend, which outperformed individual models on both CV and the Public Leaderboard.

In [ ]:
print("=" * 60)
print("FINAL ROBUST BLEND & SUBMISSION")
print("=" * 60)

# 1. Collect the exact variables you already have in memory
models = {
    'LGBM': lgb_oof_preds,
    'XGB_HighBin': xgb_oof_preds,
    'CatBoost': cb_oof_preds
}
models_test = {
    'LGBM': lgb_test_preds,
    'XGB_HighBin': xgb_test_preds,
    'CatBoost': cb_test_preds
}

for name in models:
    print(f"✅ {name} OOF AUC: {roc_auc_score(y, models[name]):.5f}")

# 2. Rank Average Blend
oof_ranks = np.array([rankdata(p) / len(y) for p in models.values()])
test_ranks = np.array([rankdata(p) / len(X_test) for p in models_test.values()])

blend_oof_rank = oof_ranks.mean(axis=0)
blend_test_rank = test_ranks.mean(axis=0)
auc_rank = roc_auc_score(y, blend_oof_rank)

# 3. Weighted Blend (Grid Search)
weight_grid = [w for w in product(np.arange(0.1, 0.85, 0.05), repeat=3) if abs(sum(w) - 1.0) < 0.001]
best_auc = 0
best_weights = None

for w in weight_grid:
    blended = sum(wi * oof_ranks[i] for i, wi in enumerate(w))
    auc = roc_auc_score(y, blended)
    if auc > best_auc:
        best_auc = auc
        best_weights = w

print("-" * 45)
print(f"Rank Average OOF:   {auc_rank:.5f}")
print(f"Weighted Blend OOF: {best_auc:.5f}")
print(f"Best Weights: LGB={best_weights[0]:.2f}, XGB={best_weights[1]:.2f}, CB={best_weights[2]:.2f}")

# 4. Create BOTH submissions so you can test
# A: Rank Average Submission
final_rank = (blend_test_rank - blend_test_rank.min()) / (blend_test_rank.max() - blend_test_rank.min())
sub_rank = pd.DataFrame({'id': test['id'], 'satisfaction': final_rank})
sub_rank.to_csv('submission_rank_average.csv', index=False)
print("\n✅ Saved: submission_rank_average.csv")

# B: Weighted Blend Submission
blend_test_weighted = sum(wi * test_ranks[i] for i, wi in enumerate(best_weights))
final_weighted = (blend_test_weighted - blend_test_weighted.min()) / (blend_test_weighted.max() - blend_test_weighted.min())
sub_weighted = pd.DataFrame({'id': test['id'], 'satisfaction': final_weighted})
sub_weighted.to_csv('submission_weighted_blend.csv', index=False)
print("✅ Saved: submission_weighted_blend.csv")

## 6. Conclusion & Future Work

- **Route Geometry** proved to be the most impactful feature engineering step, boosting the AUC by over +0.0013.
- **Next Steps**: 
  1. Incorporating the original Invistico Airline dataset as "Teacher Logits".
  2. Training deep learning models (MLPs with Entity Embeddings) to add architectural diversity.
  3. Exploring pseudo-labeling on high-confidence test predictions.